In [1]:
from workflow_auto_assembler import WorkflowAutoAssembler, AssembledWorkflow, create_function_item

### 1. Define available tools

In [2]:
from typing import Type
from pydantic import BaseModel, Field

# --- Example usage ---

# Define mock functions and their associated Pydantic models:

# 1. get_weather function


class GetWeatherInput(BaseModel):
    city: str = Field(..., description="Name of the city for which weather to be extracted.")

class GetWeatherOutput(BaseModel):
    condition: str = Field(..., description="Weather condition in the requested city.")
    temperature: float = Field(..., description="Termperature in C in the requested city.")
    humidity: float = Field(None, description="Name of the city for which weather to be extracted.")

def get_weather(inputs: GetWeatherInput) -> GetWeatherOutput:
    """Get the current weather for a city from weather forcast api."""
    return GetWeatherOutput(
        condition = "Sunny",
        temperature = 20,
        humidity = 0.6
    )

# 2. send_report_email function

class EmailInformationPoint(BaseModel):
    title: str = Field(None, description="Few word description of the information.")
    content: str = Field(..., description="Content of the information.")

class SendReportEmailInput(BaseModel):
    city: str = Field(..., description="Name of the city where report will be send.")
    information: list[EmailInformationPoint]

class SendReportEmailOutput(BaseModel):
    email_sent: bool = Field(..., description="Conformation that email was send successfully.")
    message: str = Field(None, description="Optional comments from the process.")

def send_report_email(inputs: SendReportEmailInput) -> SendReportEmailOutput:
    """Sends a report email with given information points to a city."""
    return SendReportEmailOutput(
        email_sent = True,
        message = "Email sent to city of your choosing!"
    )

# 3. query_database function

class QueryDatabaseInput(BaseModel):
    topic: str = Field(..., description="Topic of a requested piece of information.")
    location: str = Field(None, description="Filter for location name.")
    uid: str = Field(None, description="Filter for unique indentifier of the database item.")

class QueryDatabaseOutput(BaseModel):
    info: str = Field(..., description="Content of the information.")
    uid: str = Field(None, description="Unique indentifier of the database item.")

def query_database(inputs : QueryDatabaseInput) -> QueryDatabaseOutput:
    """Get information from the database with provided filters."""
    return QueryDatabaseOutput(
        info = "Content extracted from the database for your query is ...",
        uid = "0000"
    )

# Create structured items for each function
available_functions = [
    create_function_item(get_weather, GetWeatherInput, GetWeatherOutput),
    create_function_item(send_report_email, SendReportEmailInput, SendReportEmailOutput),
    create_function_item(query_database, QueryDatabaseInput, QueryDatabaseOutput)
]

available_callables = {
    "get_weather" : get_weather,
    "send_report_email" : send_report_email,
    "query_database" : query_database
}

### 2. Define task, expected inputs and outputs

In [3]:
task_description = """Query database to find information on birds and get latest weather for the city, then send an email there."""

class WfInputs(BaseModel):
    city: str = Field(..., description="Name of the city for which weather to be extracted.")

class WfOutputs(BaseModel):
    city: str = Field(..., description="Name of the city for which weather was extracted.")
    information: list[EmailInformationPoint] = Field(default=[], description="Information sent via email.")

### 3. Initialize and run workflow assembler

In [4]:
AssembledWorkflow.model_fields

{'planning': FieldInfo(annotation=Union[PlanningStepsResp, NoneType], required=False, default=PlanningStepsResp(planner=None, adaptor=None, tester=None, planner_rerun_needed=True, adaptor_rerun_needed=True, testing_errors=[], test_retries=0), description='Responses from planning steps.'),
 'workflow_completed': FieldInfo(annotation=Union[bool, NoneType], required=False, default=False, description='Indicates if workflow was completed in the preset amount of retries.'),
 'workflow': FieldInfo(annotation=Union[dict, NoneType], required=False, default=None, description='Planned and tested workflow.'),
 'description': FieldInfo(annotation=Union[WorfklowDescription, NoneType], required=False, default=WorfklowDescription(task_description=None, input_model=None, output_model=None), description='Workflow description.')}

In [5]:
wa = WorkflowAutoAssembler(
    available_functions = available_functions,
    available_callables = available_callables,
    llm_handler_params = {
        "llm_h_type" : "ollama",
        "llm_h_params" : {
            "connection_string": "http://localhost:11434",
            "model_name": "gpt-oss:20b"
        }
    }
)

wf_obj = await wa.plan_workflow(
    task_description = task_description,
    input_model = WfInputs,
    output_model = WfOutputs,
    test_inputs = WfInputs(city = "Berlin")
)

In [6]:
wf_obj.workflow_completed

True

In [7]:
wf_obj.planning.test_retries

0

In [8]:
wf_obj.workflow

[{'id': 1, 'name': 'query_database', 'args': {'topic': 'birds'}},
 {'id': 2, 'name': 'get_weather', 'args': {'city': '0.output.city'}},
 {'id': 3,
  'name': 'send_report_email',
  'args': {'city': '0.output.city',
   'information': [{'title': 'Bird Information', 'content': '1.output.info'},
    {'title': 'Weather Update', 'content': '2.output.condition'}]}},
 {'id': 4,
  'name': 'output_model',
  'args': {'city': '0.output.city',
   'information': [{'title': 'Bird Information', 'content': '1.output.info'},
    {'title': 'Weather Update', 'content': '2.output.condition'}]}}]

In [9]:
wf_obj.planning.tester.outputs

{'0': WfInputs(city='Berlin'),
 '1': QueryDatabaseOutput(info='Content extracted from the database for your query is ...', uid='0000'),
 '2': GetWeatherOutput(condition='Sunny', temperature=20.0, humidity=0.6),
 '3': SendReportEmailOutput(email_sent=True, message='Email sent to city of your choosing!'),
 '4': WfOutputs(city='Berlin', information=[EmailInformationPoint(title='Bird Information', content='Content extracted from the database for your query is ...'), EmailInformationPoint(title='Weather Update', content='Sunny')])}

### 4. Run assembled workflow

In [10]:

output = await wa.run_workflow(
    workflow_object = wf_obj,
    run_inputs = WfInputs(city = "London")
)

output.model_dump()

{'city': 'London',
 'information': [{'title': 'Bird Information',
   'content': 'Content extracted from the database for your query is ...'},
  {'title': 'Weather Update', 'content': 'Sunny'}]}